In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
!ls /kaggle/input/q1-stage-3-2026/PlantVillage

In [ ]:
# Write your code here
from torchvision import transforms
import os
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

transform_train = transforms.Compose([ transforms.RandomRotation(15) ,
                                      transforms.Resize((32, 32)),
                                      transforms.ToTensor() ])

transform_test = transforms.Compose([transforms.Resize((32, 32)),
                                      transforms.ToTensor() ])

train_dir = os.path.join(path, "PlantVillage","train")
test_dir = os.path.join(path, "PlantVillage","test")

train_dataset = ImageFolder(root=train_dir, transform=transform_train)
test_dataset  = ImageFolder(root=test_dir,  transform=transform_test)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

trainloader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
testloader = DataLoader(test_dataset, batch_size=100, shuffle=False, num_workers=2)

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import torch

# Create a function to visualize samples
def visualize_samples(dataset, num_samples=8, title="Dataset Samples"):

    indices = random.sample(range(len(dataset)), num_samples)

    cols = 4
    rows = (num_samples + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(12, 3 * rows))
    axes = axes.flatten()

    for i, idx in enumerate(indices):
        image, label = dataset[idx]
        if isinstance(image, torch.Tensor):
            image = image.permute(1, 2, 0).numpy()
        class_name = dataset.classes[label]
        axes[i].imshow(image)
        axes[i].set_title(f"{class_name}\n(Label: {label})", fontsize=10)
        axes[i].axis('off')
    for i in range(num_samples, len(axes)):
        axes[i].axis('off')

    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_samples(train_dataset, num_samples=8, title="Training Dataset Samples")
visualize_samples(test_dataset, num_samples=8, title="Testing Dataset Samples")

In [ ]:
# Write your code here
import torch.nn as nn

class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv4 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv5 = nn.Conv2d(128, 256, 3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(16728, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x=self.conv1(x)
        x=nn.BatchNorm2d(16)
        x = self.pool(self.relu(x))

        x=self.conv2(x)
        x=nn.BatchNorm2d(32)
        x = self.pool(self.relu(x))

        x=self.conv3(x)
        x=nn.BatchNorm2d(64)
        x = self.pool(self.relu(x))

        x=self.conv4(x)
        x=nn.BatchNorm2d(128)
        x = self.pool(self.relu(x))

        x=self.conv5(x)
        x=nn.BatchNorm2d(256)
        x = self.pool(self.relu(x))


        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        return self.fc2(x)

In [ ]:
# Write your code here
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train() # Set the model to training mode
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad() # Zero the parameter gradients

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

    epoch_loss = running_loss / total_samples
    epoch_accuracy = correct_predictions / total_samples
    return epoch_loss, epoch_accuracy

def validate_epoch(model, dataloader, criterion, device):
    model.eval() # Set the model to evaluation mode
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad(): # Disable gradient calculation
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()

    epoch_loss = running_loss / total_samples
    epoch_accuracy = correct_predictions / total_samples
    return epoch_loss, epoch_accuracy

In [ ]:
# Write your code here
import torch.optim as optim
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model=CNNModel().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 5

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

print("Starting Training...")
for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, trainloader, criterion, optimizer, device)
    val_loss, val_acc = validate_epoch(model, testloader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    print(f'Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

print("Finished Training.")

In [ ]:
# Write your code here
